# YOLOv11 — Shelf Void Detection (Free GPU Training)

**Roman Urdu guide (beginner ke liye):**

Yeh notebook aapke Roboflow dataset zip se **free me** YOLOv11 model train karta hai — Google Colab ke free T4 GPU pe. Zero paise, zero Roboflow credits.

## Kaise use karein (5 steps):
1. **GPU on karo** → Runtime menu → *Change runtime type* → **T4 GPU** → Save
2. **dataset.zip upload karo** → left sidebar 📁 → `/content` → Upload → `dataset.zip`
3. **Run All** karo (Ctrl+F9 ya Runtime → Run all)
4. **Wait** ~25-35 min (yolo11s, 120 epochs)
5. **best.pt download** → niche wali cells automatically download karenge

## Classes
- `0 = void` (khali shelf / empty space)

> **New!** Dataset me ab `506 images` hain with `2,342 void annotations` (oversample ×3 = 1,518 train images). Class imbalance fix ho gaya!

---


## Step 1 — GPU check karo

Cell run karne pe **Tesla T4** ya koi GPU dikhni chahiye. Agar `command not found` ya CPU aaye toh: Runtime → Change runtime type → T4 GPU.

In [ ]:
!nvidia-smi
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - GPU on karo!")


## Step 2 — Ultralytics (YOLOv11) install karo

Official YOLOv11 package. Install me ~1 min lagta hai.

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
import os, shutil, yaml, glob
print("Ultralytics installed")


## Step 3 — Dataset upload + setup

### Pehle zip upload karo:
Left sidebar me **folder icon 📁** → `/content` → **Upload** → apna file select karo. **Naam `dataset.zip` rakhna** (ya neeche `ZIP_NAME` badal do).

Upload ~5 min lagta hai. Upload complete hone ke baad hi yeh cell run karo.

**Download kahan se:** Roboflow → aapka project → Version → *Download Dataset* → **YOLOv11 PyTorch** format.

In [ ]:
ZIP_NAME = "dataset.zip"   # agar file ka naam alag hai toh yahan badal do
ROOT = "/content"
DATA_DIR = ROOT + "/shelf_dataset"

# Clean previous
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)

zip_path = os.path.join(ROOT, ZIP_NAME)
assert os.path.exists(zip_path), zip_path + " nahi mila! Pehle zip upload karo (sidebar 📁 se Upload)."

# Unzip
!unzip -q "{zip_path}" -d "{DATA_DIR}"
print("Unzipped to:", DATA_DIR)
print("Contents:", os.listdir(DATA_DIR))


In [ ]:
# Structure check + data.yaml fix
yaml_files = glob.glob(DATA_DIR + "/**/data.yaml", recursive=True)
assert yaml_files, "data.yaml nahi mila"
data_yaml_path = os.path.abspath(yaml_files[0])
base_dir = os.path.dirname(data_yaml_path)

cfg = {
    "train": base_dir + "/train/images",
    "val":   base_dir + "/valid/images",
    "test":  base_dir + "/test/images",
    "nc": 1,
    "names": ["void"],
}
with open(data_yaml_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("data.yaml fixed:", data_yaml_path)
print(open(data_yaml_path).read())

for split in ["train", "valid", "test"]:
    imgs = glob.glob(base_dir + "/" + split + "/images/*.jpg")
    labs = glob.glob(base_dir + "/" + split + "/labels/*.txt")
    print("%-6s: %d images, %d labels" % (split, len(imgs), len(labs)))


## Step 3.5 — Class Imbalance Fix (Oversampling) ⭐

Ab dataset me **2,344 void annotations** hain. Unhe 3x oversample karke training data boost karte hain.


In [ ]:
# === OVERSAMPLING: void (missing) images ko duplicate karo ===
OVERSAMPLE_FACTOR = 3   # void image ko itni baar rakho (3 = original + 2 copies)

train_img_dir = base_dir + "/train/images"
train_lab_dir = base_dir + "/train/labels"

# Void wali images dhoondo (jinke label me class 0 = missing hai)
def has_void(lab_path):
    with open(lab_path) as f:
        return any(line.strip().startswith("0 ") for line in f)

void_imgs = []
for img in glob.glob(train_img_dir + "/*.jpg"):
    lab = os.path.join(train_lab_dir, os.path.splitext(os.path.basename(img))[0] + ".txt")
    if os.path.exists(lab) and has_void(lab):
        void_imgs.append((img, lab))

before_n = len(glob.glob(train_img_dir + "/*.jpg"))
added = 0
for img, lab in void_imgs:
    for k in range(OVERSAMPLE_FACTOR - 1):   # original pehle se hai, sirf copies add karo
        stem, ext = os.path.splitext(img)
        img_dup = stem + "_dup" + str(k) + ext
        lab_dup = os.path.splitext(lab)[0] + "_dup" + str(k) + ".txt"
        if not os.path.exists(img_dup):
            shutil.copy(img, img_dup)
            shutil.copy(lab, lab_dup)
            added += 1

after_n = len(glob.glob(train_img_dir + "/*.jpg"))
print("Void (missing) wali train images:", len(void_imgs))
print("Oversample x%d: %d void copies add kiye" % (OVERSAMPLE_FACTOR, added))
print("Train images: %d -> %d" % (before_n, after_n))


## Step 3.6 — Class Weights (Focal Loss) ⭐

Void class ko extra weight dene ke liye focal loss use karte hain. Yeh hard examples (void spaces) pe zyada focus karega.

```python
# Focal loss gamma = 1.5 (default 0.0)
# Higher gamma = hard examples pe zyada focus
```


In [ ]:
# === CLASS WEIGHTS: Focal loss for void ===
FOCAL_LOSS_GAMMA = 1.5   # default 0.0, higher = void ko zyada weight
CLS_LOSS = 0.7            # default 0.5, classification loss coefficient

print("Focal loss gamma:", FOCAL_LOSS_GAMMA)
print("Cls loss coeff:", CLS_LOSS)
print("✅ Void class ko extra weight mil raha hai (focal loss)")


## Step 4 — Training (improved settings)

Void recall badhane ke liye settings:

| Setting | Value | Kyun |
|---|---|---|
| Model | `yolo11s.pt` (small) | Achi capacity → voids behtar seekhega |
| Epochs | `120`, patience `35` | Acha converge, jaldi early-stop na ho |
| Oversampling | ✅ Step 3.5 (void ×3) | Balance theek |
| Class Weights | ✅ Step 3.6 | Void class ko extra weight |

> Training time ~25-35 min (yolo11s). Agar **`OutOfMemory (OOM)`** aaye toh niche `BATCH = 16` ko `8` kar do.
> Fast chahiye toh `MODEL_VARIANT = "yolo11n.pt"` (par void recall thoda kam).

In [ ]:
# === CONFIG ===
MODEL_VARIANT = "yolo11s.pt"   # yolo11n.pt (fast) / yolo11s.pt (better, default) / yolo11m.pt (best, slow)
EPOCHS = 120
IMG_SIZE = 640
BATCH = 16                     # OOM aaye toh 8 kar do
PATIENCE = 35
RUN_NAME = "shelf_void_v2"

print("Training: %s | %d epochs | batch %d | oversample x3" % (MODEL_VARIANT, EPOCHS, BATCH))

model = YOLO(MODEL_VARIANT)
results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    name=RUN_NAME,
    patience=PATIENCE,
    cache=True,
    plots=True,
    "    exist_ok=True,
    "    fl_gamma=FOCAL_LOSS_GAMMA,    # focal loss for void
    "    cls=CLS_LOSS,                 # classification loss weight
)
print("✅ TRAINING DONE")


## Step 5 — Results + Metrics dekho

`missing` class ka **recall** sabse important hai — yeh batata hai model khali shelf kitni baar pakad leta hai.

In [ ]:
# Validation on test set
metrics = model.val(data=data_yaml_path, split="test", plots=True)
print("\n=== OVERALL ===")
print("mAP50      : %.4f" % metrics.box.map50)
print("mAP50-95   : %.4f" % metrics.box.map)
print("Precision  : %.4f" % metrics.box.mp)
print("Recall     : %.4f" % metrics.box.mr)

print("\n=== VOID CLASS ===")
print("mAP50=%.4f  P=%.4f  R=%.4f" % (metrics.box.ap50(0), metrics.box.p[0], metrics.box.r[0]))


## Step 6 — Best Model Download

In [ ]:
best_model_path = "runs/detect/" + RUN_NAME + "/weights/best.pt"
assert os.path.exists(best_model_path), "best.pt nahi mila — training check karo"

from google.colab import files
files.download(best_model_path)
print("best.pt download ho raha hai... (browser popup)")
print("Size: %.1f MB" % (os.path.getsize(best_model_path)/1024/1024))


## Step 7 — Training Graphs

In [ ]:
from IPython.display import Image, display
run_dir = "runs/detect/" + RUN_NAME
for img_name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    p = run_dir + "/" + img_name
    if os.path.exists(p):
        print("\n--- " + img_name + " ---")
        display(Image(filename=p, width=600))


---
*Notebook • YOLOv11 • Free Colab GPU • Class-imbalance oversampling*